In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy.interpolate import interp1d

In [ ]:
# Step 1: Load and Preprocess Data (Full Time-Series)
# Assume 'amaranthus_data.csv' with 8441 rows at 5-min intervals
df = pd.read_csv('Smart-Agriculture-Business-Application-with-Multi-Crop-Support/amaranthus_dataset_1_aligned.csv')

# Convert timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date
df['datetime'] = df['timestamp']  # For sorting

# Sort by time
df = df.sort_values('datetime').reset_index(drop=True)

# Handle N/A (impute with forward fill, then mean)
numeric_cols = ['air_temp_c', 'humidity_percent', 'water_soil_temp_c', 'air_pressure_hpa',
                'soil_moisture_raw', 'light_level_raw', 'rain_level_raw', 'co2_ppm',
                'nh3_ppm', 'voc_ppm', 'nitrogen_mg_kg', 'phosphorus_mg_kg', 'potassium_mg_kg']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(method='ffill').fillna(df[col].mean())

# Aggregate to HOURLY for LSTM (reduce from 5-min to manageable seq; ~144 rows/day *30=~4320)
hourly_df = df.groupby(pd.Grouper(key='datetime', freq='H')).agg({
    'air_temp_c': 'mean',
    'humidity_percent': 'mean',
    'soil_moisture_raw': 'mean',
    'light_level_raw': 'mean',
    'rain_level_raw': 'sum',  # Hourly rain sum
    'nitrogen_mg_kg': 'mean',
    'phosphorus_mg_kg': 'mean',
    'potassium_mg_kg': 'mean'
}).reset_index()

# Add stress features (computed on hourly)
hourly_df['temp_stress'] = np.abs(hourly_df['air_temp_c'] - 25) / 25
hourly_df['hum_stress'] = np.abs(hourly_df['humidity_percent'] - 75) / 75
hourly_df['moist_stress'] = np.abs(hourly_df['soil_moisture_raw'] - 50000) / 50000
hourly_df['light_bonus'] = np.clip(hourly_df['light_level_raw'] / 800, 0, 2)
hourly_df['rain_penalty'] = np.clip(hourly_df['rain_level_raw'] / 10000, 0, 1)
hourly_df['n_stress'] = np.abs(hourly_df['nitrogen_mg_kg'] - 20) / 20

# Step 1.5: Incorporate Heights (Interpolate to hourly; assume constant between measurements)
planting_date = datetime(2025, 6, 19)
height_measurements = {
    datetime(2025, 6, 29): 5.0,
    datetime(2025, 7, 8): 15.0,
    datetime(2025, 7, 18): 30.5,
    datetime(2025, 7, 22): 45.0
}
height_dates = sorted(height_measurements.keys())
height_values = list(height_measurements.values())
days_from_plant = [(d - planting_date).total_seconds() / 3600 for d in height_dates]  # Hours from planting
interp_func = interp1d(days_from_plant, height_values, kind='linear', fill_value='extrapolate')
hourly_df['hours_from_planting'] = (hourly_df['datetime'] - planting_date).dt.total_seconds() / 3600
hourly_df['height_cm'] = interp_func(hourly_df['hours_from_planting'].values)
hourly_df['growth_rate_cm_hour'] = hourly_df['height_cm'].diff().fillna(0)  # Hourly growth

# Targets: Yield 90g/plant, harvest 33 days (792 hours); repeat for all rows in period
total_hours = 33 * 24
hourly_df['yield_g_per_plant'] = 90.0
hourly_df['harvest_hours'] = total_hours

# Feature columns for LSTM
feature_cols = ['air_temp_c', 'humidity_percent', 'soil_moisture_raw', 'light_level_raw',
                'rain_level_raw', 'nitrogen_mg_kg', 'phosphorus_mg_kg', 'potassium_mg_kg',
                'temp_stress', 'hum_stress', 'moist_stress', 'light_bonus', 'rain_penalty', 'n_stress',
                'height_cm', 'growth_rate_cm_hour']

In [ ]:
# Step 2: Simulate Time-Series Data for Training (200 fields, ~30 days hourly)
np.random.seed(42)
n_fields = 200
hours_per_day = 24
n_days = 30
n_hours = n_days * hours_per_day
sim_data = []

base_growth_rate_cm_day = 1.36  # From your data
height_yield_factor = 2.0  # g/cm
pest_factor = np.random.uniform(0.8, 1.0)  # Per field

for field in range(n_fields):
    base_temp = 25 + np.random.normal(0, 3)
    base_hum = 75 + np.random.normal(0, 10)
    base_moist = 50000 + np.random.normal(0, 10000)
    base_light = 800 + np.random.normal(0, 200)
    base_rain_hourly = np.random.uniform(0, 200)  # Low hourly rain
    base_n = 20 + np.random.normal(0, 5)
    base_p = 50 + np.random.normal(0, 10)
    base_k = 100 + np.random.normal(0, 20)
    pest_factor = np.random.uniform(0.8, 1.0)
    
    current_height = 0.0
    total_stress = 0.0
    
    for h in range(n_hours):
        # Hourly variations
        temp = base_temp + np.random.normal(0, 0.5)
        hum = base_hum + np.random.normal(0, 2)
        moist = base_moist + np.random.normal(0, 1000)
        light = base_light + np.random.normal(0, 20)
        rain = base_rain_hourly + np.random.normal(0, 50)
        n = base_n
        
        # Stress
        t_stress = np.abs(temp - 25) / 25
        h_stress = np.abs(hum - 75) / 75
        m_stress = np.abs(moist - 50000) / 50000
        n_stress = np.abs(n - 20) / 20
        daily_stress = (t_stress + h_stress + m_stress + n_stress) / 4
        total_stress += daily_stress
        avg_stress = total_stress / (h + 1)
        
        l_bonus = np.clip(light / 800, 0, 2)
        r_penalty = np.clip(rain / 10000, 0, 1)
        hourly_growth = (base_growth_rate_cm_day / 24) * l_bonus * (1 - daily_stress) * (1 - r_penalty)
        current_height += max(0, hourly_growth)
        
        sim_yield = current_height * height_yield_factor * pest_factor if h == n_hours - 1 else np.nan
        sim_harvest = n_hours * np.exp(0.05 * avg_stress / n_hours)  # Adjusted
        
        row = {
            'datetime': planting_date + timedelta(hours=h),
            'hours_from_planting': h,
            'air_temp_c': temp, 'humidity_percent': hum, 'soil_moisture_raw': moist,
            'light_level_raw': light, 'rain_level_raw': rain,
            'nitrogen_mg_kg': n, 'phosphorus_mg_kg': base_p, 'potassium_mg_kg': base_k,
            'temp_stress': t_stress, 'hum_stress': h_stress, 'moist_stress': m_stress,
            'light_bonus': l_bonus, 'rain_penalty': r_penalty, 'n_stress': n_stress,
            'height_cm': current_height, 'growth_rate_cm_hour': hourly_growth,
            'yield_g_per_plant': sim_yield, 'harvest_hours': sim_harvest
        }
        sim_data.append(row)

sim_df = pd.DataFrame(sim_data)
sim_df['yield_g_per_plant'] = sim_df.groupby('datetime')['yield_g_per_plant'].ffill().fillna(method='bfill')
sim_df['harvest_hours'] = sim_df.groupby('datetime')['harvest_hours'].ffill().fillna(method='bfill')

# Combine: full hourly sequences
full_df = pd.concat([hourly_df[feature_cols + ['yield_g_per_plant', 'harvest_hours']], sim_df], ignore_index=True)
full_df = full_df.sort_values('datetime').reset_index(drop=True)  # Global sort for TS CV

# Normalize features (MinMax for LSTM)
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
full_df[feature_cols] = scaler.fit_transform(full_df[feature_cols])

In [ ]:
# Step 3: Prepare Sequences for LSTM (Window size 24 hours -> predict next 24h avg yield/harvest)
seq_length = 24  # 1 day window
def create_sequences(data, seq_len):
    X, y_yield, y_harvest = [], [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y_yield.append(data[i+seq_len, feature_cols.index('yield_g_per_plant')])  # Wait, targets separate
    Wait, better: X from features, y from targets
Wait, fix:
X_seq = []
y_yield_seq = []
y_harvest_seq = []
for i in range(len(full_df) - seq_length):
    X_seq.append(full_df[feature_cols].iloc[i:i+seq_length].values)
    y_yield_seq.append(full_df['yield_g_per_plant'].iloc[i+seq_length])
    y_harvest_seq.append(full_df['harvest_hours'].iloc[i+seq_length])

X_seq = np.array(X_seq)
y_yield_seq = np.array(y_yield_seq).reshape(-1, 1)
y_harvest_seq = np.array(y_harvest_seq).reshape(-1, 1)

# PyTorch Dataset
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y_yield, y_harvest):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y_yield = torch.tensor(y_yield, dtype=torch.float32)
        self.y_harvest = torch.tensor(y_harvest, dtype=torch.float32)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y_yield[idx], self.y_harvest[idx]

# LSTM Model
class LSTMPredictor(nn.Module):
    def __init__(self, input_size, hidden_size=50, num_layers=1, output_size=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc_yield = nn.Linear(hidden_size, output_size)
        self.fc_harvest = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        yield_out = self.fc_yield(hn[-1])
        harvest_out = self.fc_harvest(hn[-1])
        return yield_out, harvest_out

input_size = len(feature_cols)
model = LSTMPredictor(input_size)

# Loss and Optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Step 4: Cross-Validation (TimeSeriesSplit for non-overlap)
tscv = TimeSeriesSplit(n_splits=5)
cv_r2_yield = []
cv_r2_harvest = []

for train_idx, val_idx in tscv.split(X_seq):
    train_dataset = TimeSeriesDataset(X_seq[train_idx], y_yield_seq[train_idx], y_harvest_seq[train_idx])
    val_dataset = TimeSeriesDataset(X_seq[val_idx], y_yield_seq[val_idx], y_harvest_seq[val_idx])
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    
    # Train model
    model.train()
    for epoch in range(10):  # Short epochs for CV
        for X_batch, y_y_batch, y_h_batch in train_loader:
            optimizer.zero_grad()
            y_y_pred, y_h_pred = model(X_batch)
            loss_y = criterion(y_y_pred, y_y_batch)
            loss_h = criterion(y_h_pred, y_h_batch)
            loss = loss_y + loss_h
            loss.backward()
            optimizer.step()
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        val_y_y_pred, val_y_h_pred = model(torch.tensor(X_seq[val_idx], dtype=torch.float32))
        r2_y = r2_score(y_yield_seq[val_idx], val_y_y_pred.numpy())
        r2_h = r2_score(y_harvest_seq[val_idx], val_y_h_pred.numpy())
    
    cv_r2_yield.append(r2_y)
    cv_r2_harvest.append(r2_h)

print(f"Yield CV R²: {np.mean(cv_r2_yield):.3f} (+/- {np.std(cv_r2_yield)*2:.3f})")
print(f"Harvest CV R²: {np.mean(cv_r2_harvest):.3f} (+/- {np.std(cv_r2_harvest)*2:.3f})")

# Train Final Model on Full Data
full_dataset = TimeSeriesDataset(X_seq, y_yield_seq, y_harvest_seq)
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=False)

model.train()
for epoch in range(50):  # More epochs
    for X_batch, y_y_batch, y_h_batch in full_loader:
        optimizer.zero_grad()
        y_y_pred, y_h_pred = model(X_batch)
        loss_y = criterion(y_y_pred, y_y_batch)
        loss_h = criterion(y_h_pred, y_h_batch)
        loss = loss_y + loss_h
        loss.backward()
        optimizer.step()

In [ ]:
# Step 5: Predict on Your Full 30-Day Sequence (Use last seq_length hours as input for final prediction)
# Assume hourly_df is your real data (~720 hours for 30 days)
your_seq = hourly_df[feature_cols].tail(seq_length).values  # Last day as proxy for ongoing; or full avg seq
your_seq = scaler.transform(your_seq)  # Ensure scaled
your_X = torch.tensor(your_seq.reshape(1, seq_length, -1), dtype=torch.float32)

model.eval()
with torch.no_grad():
    pred_yield_t, pred_harvest_t = model(your_X)
    pred_yield = pred_yield_t.item() * 14  # Total for 14 plants (denormalize if needed; here raw)
    pred_harvest_days = pred_harvest_t.item() / 24

print(f"Predicted Total Yield (LSTM on full seq): {pred_yield:.1f} g (for 14 plants)")
print(f"Predicted Harvest Time: {pred_harvest_days:.1f} days")

# For full period avg input (as fallback)
period_avg = hourly_df[feature_cols].mean().values.reshape(1, 1, -1)  # But LSTM needs seq; pad or use RF hybrid
# Note: For avg, could use last code's RF, but here focus LSTM seq

# Visualization: Predicted vs Actual on sim
model.eval()
with torch.no_grad():
    all_pred_y, all_pred_h = model(torch.tensor(X_seq, dtype=torch.float32))
    all_pred_y = all_pred_y.numpy().flatten()
    all_pred_h = all_pred_h.numpy().flatten()

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(y_yield_seq, all_pred_y)
plt.plot([0, 150], [0, 150], 'r--')
plt.xlabel('Actual Yield')
plt.ylabel('Predicted Yield')
plt.title('LSTM Yield Pred vs Actual')

plt.subplot(1, 2, 2)
plt.plot(hourly_df['hours_from_planting']/24, hourly_df['height_cm'], 'g-', label='Height Growth')
plt.xlabel('Days')
plt.ylabel('Height (cm)')
plt.title('Your Field Growth')
plt.legend()

plt.tight_layout()
plt.show()

# Save model: torch.save(model.state_dict(), 'lstm_amaranthus.pth')